In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from toxic_comments.config import HEAVY_TEXT_COLUMN, LABEL_COLUMNS, LIGHT_TEXT_COLUMN, TEXT_COLUMN
from toxic_comments.cleaning import audit_dataset, clean_heavy, clean_light, process_cleaning, validate_training_data

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

PosixPath('/home/jkl0909/minhhuyen/toxic-comments')

In [2]:
data_path = RAW_DIR / "train.csv"

raw_data = pd.read_csv(data_path)
validated_data = validate_training_data(raw_data)

print("Required label columns:")
print(LABEL_COLUMNS)
print("\nDtypes after validation:")
print(validated_data[[TEXT_COLUMN, *LABEL_COLUMNS]].dtypes)

audit_dataset(validated_data)

Required label columns:
['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

Dtypes after validation:
comment_text     object
toxic             int64
severe_toxic      int64
obscene           int64
threat            int64
insult            int64
identity_hate     int64
dtype: object


{'rows': 159571,
 'null_comment_text': 0,
 'empty_comment_text': 0,
 'duplicate_ids': 0,
 'duplicate_texts': 0,
 'label_counts': {'toxic': 15294,
  'severe_toxic': 1595,
  'obscene': 8449,
  'threat': 478,
  'insult': 7877,
  'identity_hate': 1405}}

In [3]:
sample_text = "[[User:Example]] You're f.u.c.k.i.n.g annoying!!!! Visit https://example.com 12:30, 1 January 2020 (UTC)"

print("RAW:")
print(sample_text)
print("\nLIGHT:")
print(clean_light(sample_text))
print("\nHEAVY:")
print(clean_heavy(clean_light(sample_text)))

RAW:
[[User:Example]] You're f.u.c.k.i.n.g annoying!!!! Visit https://example.com 12:30, 1 January 2020 (UTC)

LIGHT:
You're f.u.c.k.i.n.g annoying!!! Visit

HEAVY:
you fucking annoying visit


In [4]:
cleaned_data = process_cleaning(
    validated_data,
    name=data_path.name,
    is_train=True,
    drop_duplicate_rows=True,
    drop_empty_light_train_rows=True,
    verbose=True,
)

print(f"Cleaned shape: {cleaned_data.shape}")
cleaned_data.head()


>>> train.csv (159,571 rows)
    cleaning level 1 (light) ...
    cleaning level 2 (heavy) ...
    75 rows empty in HEAVY but fine in LIGHT (non-Latin / emoji) -> KEPT, flagged as is_non_latin
    dropped 95 rows with no usable text at all
    170 rows unusable for bag-of-words -> filter on is_empty_heavy for TF-IDF
Cleaned shape: (159476, 28)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,comment_light,comment_heavy,...,punct_ratio,n_newlines,n_urls,n_ips,n_you,n_masked_words,has_shouting,is_empty_light,is_empty_heavy,is_non_latin
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0,Explanation Why the edits made under my userna...,explanation why edits made username hardcore m...,...,0.037879,1,0,1,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0,D'aww! He matches this background colour I'm s...,aww matches background colour seemingly stuck ...,...,0.107143,0,0,0,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0,"Hey man, I'm really not trying to edit war. It...",hey man really not trying edit war just guy co...,...,0.025751,0,0,0,0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0,""" More I can't make any real suggestions on im...",cannot make real suggestions improvement wonde...,...,0.030547,4,0,0,1,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0,"You, sir, are my hero. Any chance you remember...",you sir hero chance you remember page,...,0.074627,0,0,0,2,0,0,0,0,0


In [5]:
text_columns = [
    TEXT_COLUMN,
    LIGHT_TEXT_COLUMN,
    HEAVY_TEXT_COLUMN,
    "is_empty_light",
    "is_empty_heavy",
    "is_non_latin",
]

cleaned_data[text_columns].head(10)

,comment_text,comment_light,comment_heavy,is_empty_light,is_empty_heavy,is_non_latin
0,Explanation\nWhy the edits made under my usern...,Explanation Why the edits made under my userna...,explanation why edits made username hardcore m...,0,0,0
1,D'aww! He matches this background colour I'm s...,D'aww! He matches this background colour I'm s...,aww matches background colour seemingly stuck ...,0,0,0
2,"Hey man, I'm really not trying to edit war. It...","Hey man, I'm really not trying to edit war. It...",hey man really not trying edit war just guy co...,0,0,0
3,"""\nMore\nI can't make any real suggestions on ...",""" More I can't make any real suggestions on im...",cannot make real suggestions improvement wonde...,0,0,0
4,"You, sir, are my hero. Any chance you remember...","You, sir, are my hero. Any chance you remember...",you sir hero chance you remember page,0,0,0
5,"""\n\nCongratulations from me as well, use the ...",""" Congratulations from me as well, use the too...",congratulations well use tools well talk,0,0,0
6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,cocksucker you piss around work,0,0,0
7,Your vandalism to the Matt Shirvington article...,Your vandalism to the Matt Shirvington article...,your vandalism matt shirvington article revert...,0,0,0
8,Sorry if the word 'nonsense' was offensive to ...,Sorry if the word 'nonsense' was offensive to ...,sorry word nonsense offensive you anyway not i...,0,0,0
9,alignment on this subject and which are contra...,alignment on this subject and which are contra...,alignment subject contrary dulithgow,0,0,0


In [6]:
feature_columns = [
    "n_chars",
    "n_words",
    "n_unique_words",
    "unique_word_ratio",
    "mean_word_len",
    "caps_ratio",
    "n_exclaim",
    "n_question",
    "punct_ratio",
    "n_newlines",
    "n_urls",
    "n_ips",
    "n_you",
    "n_masked_words",
    "has_shouting",
]

cleaned_data[feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
n_chars,159476.0,394.273408,590.835053,6.0000,96.000000,205.000000,436.000000,5000.000000
n_words,159476.0,67.308316,99.249297,1.0000,17.000000,36.000000,75.000000,1411.000000
n_unique_words,159476.0,48.121247,54.443107,1.0000,16.000000,31.000000,59.000000,816.000000
unique_word_ratio,159476.0,0.854947,0.128162,0.0008,0.779070,0.875000,0.956522,1.000000
mean_word_len,159476.0,4.933644,7.202670,1.0000,4.333333,4.696970,5.105263,1655.000000
caps_ratio,159476.0,0.051546,0.092818,0.0000,0.020690,0.031746,0.048780,0.998189
n_exclaim,159476.0,0.661930,25.907255,0.0000,0.000000,0.000000,0.000000,4942.000000
n_question,159476.0,0.449510,1.590311,0.0000,0.000000,0.000000,1.000000,209.000000
punct_ratio,159476.0,0.047455,0.037964,0.0000,0.026470,0.038462,0.056701,0.994366
n_newlines,159476.0,2.522229,5.963711,0.0000,0.000000,1.000000,2.000000,312.000000


In [7]:
comparison = cleaned_data[[TEXT_COLUMN, LIGHT_TEXT_COLUMN, HEAVY_TEXT_COLUMN]].sample(
    min(10, len(cleaned_data)),
    random_state=42,
)

pd.set_option("display.max_colwidth", 180)
comparison

,comment_text,comment_light,comment_heavy
68263,"Wizardman is finally a person that realised exactly how bossy, rude and childish you behave. I don't need any diffs to prove that, the RFC has a large collection of your consta...","Wizardman is finally a person that realised exactly how bossy, rude and childish you behave. I don't need any diffs to prove that, the RFC has a large collection of your consta...",wizardman finally person realised exactly bossy rude childish you behave not need diffs prove rfc large collection your constant breach civility hope people realise so you fina...
88847,"SuperHamster is Super Pathetic \n\nHey everybody, SuperHamster here. I'm really lonely and sad and pathetic mostly because I have no friends and spend all my time vandalizing ...","SuperHamster is Super Pathetic Hey everybody, SuperHamster here. I'm really lonely and sad and pathetic mostly because I have no friends and spend all my time vandalizing Wiki ...",superhamster super pathetic hey everybody superhamster really lonely sad pathetic mostly no friends spend time vandalizing wiki pages
116823,Thanks for reverting recently damaged articles.,Thanks for reverting recently damaged articles.,thanks reverting recently damaged articles
55728,"Excuse me, but could you explain this statement, please? The person who removed the link persists in mixing up the link to the performances and the link to the specific edition...","Excuse me, but could you explain this statement, please? The person who removed the link persists in mixing up the link to the performances and the link to the specific edition...",excuse could you explain statement please person who removed link persists mixing link performances link specific edition also removed different wikipedia page wikipedia page e...
113314,"""\nAnd finally to seal my argument this is the credits of the film from the offical website. Jack In """,""" And finally to seal my argument this is the credits of the film from the offical website. Jack In """,finally seal argument credits film offical website jack
72860,and the multiple objections from many experianced contributors,and the multiple objections from many experianced contributors,multiple objections many experianced contributors
53339,Hardcore Punk \n\nhey thanks for the removal of sourced material . it was fun to put it back and will be fun to continue putting it back . hope all is well . 68.39.152.45,Hardcore Punk hey thanks for the removal of sourced material . it was fun to put it back and will be fun to continue putting it back . hope all is well .,hardcore punk hey thanks removal sourced material fun put back fun continue putting back hope well
92648,"Back again. \n\nI was just told off for cyber vandalism. Amazing. All I asked was a serious question regarding your seemingly endless amount of free time, and Im labelled a cy...","Back again. I was just told off for cyber vandalism. Amazing. All I asked was a serious question regarding your seemingly endless amount of free time, and Im labelled a cyber v...",back just told cyber vandalism amazing asked serious question regarding your seemingly endless amount free time im labelled cyber vigilante perhaps you move your mams house man...
59996,"""\n\nCut-and-paste\nChange to more polite and good word\nThe words of this section heading are taken from 's edit summary here. The words of in this diff are constructive and...",""" Cut-and-paste Change to more polite and good word The words of this section heading are taken from 's edit summary here. The words of in this diff are constructive and welcom...",cut paste change polite good word words section heading taken edit summary words diff constructive welcome reasonable copy paste sentences talk page venue maritime boundary hey...
56016,"PaxEquilibrium \n\nI labled you're edit as vandalism because I felt it was in the strictest sense. You were attempting to degrade the article with POV and unofficial claims, I ...","PaxEquilibrium I labled you're ed

In [8]:
output_csv = PROCESSED_DIR / "train_clean.csv"
cleaned_data.to_csv(output_csv, index=False)

print(f"Saved: {output_csv}")
print(f"Rows: {len(cleaned_data):,}")
print(f"Columns: {len(cleaned_data.columns):,}")

Saved: /home/jkl0909/minhhuyen/toxic-comments/data/processed/train_clean.csv
Rows: 159,476
Columns: 28
